# baseline

이 베이스라인 코드는 `사전학습 모델 로드`, `배치 학습`, `파인튜닝`, `양자화`, `PEFT` 등이 적용된 버전입니다.

윈도우 데스크탑의 RTX 5060 ti GPU 환경에서 개발되었습니다.

# 환경 준비

개발 환경에 필요한 라이브러리 버전을 고정하고 최신 버전으로 라이브러리를 업데이트합니다.

In [ ]:
# ================================================================
# [1단계] PyTorch Nightly 설치 (CUDA 12.8 지원 버전)
# ================================================================
# RTX 5060 Ti는 최신 GPU(Blackwell 아키텍처)이므로,
# 안정 버전(stable)이 아닌 nightly(개발 중) 버전의 PyTorch가 필요합니다.
# --pre : 프리릴리즈(사전 배포) 버전을 설치하겠다는 플래그
# --index-url : PyTorch 공식 nightly 빌드 저장소(cu128 = CUDA 12.8)
# torchvision : 이미지 전처리/모델용, torchaudio : 오디오용 (의존성)
!pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu128


In [1]:
# ================================================================
# [검증] PyTorch 설치 상태 및 GPU 인식 여부 확인
# ================================================================
# 이 셀을 실행하여 아래 3가지가 정상 출력되는지 확인하세요:
#   1) torch.__version__  → 설치된 PyTorch 버전 (예: 2.7.0.dev...)
#   2) torch.cuda.is_available() → True여야 GPU 사용 가능
#   3) torch.cuda.get_device_name() → GPU 이름 (예: NVIDIA GeForce RTX 5060 Ti)
# 만약 False가 출력되면 CUDA 드라이버 또는 PyTorch 설치를 다시 확인하세요.
import torch

print(torch.__version__)          # PyTorch 버전
print(torch.cuda.is_available())  # GPU 사용 가능 여부 (True/False)
print(torch.cuda.get_device_name())  # 인식된 GPU 이름


2.11.0+cu128
True
NVIDIA GeForce RTX 5060 Ti


In [3]:
# ================================================================
# [1단계] 핵심 라이브러리 설치
# ================================================================
# -q : quiet 모드 (설치 로그를 간결하게 출력)
# --upgrade : 이미 설치된 패키지도 최신 호환 버전으로 업그레이드
#
# 각 라이브러리 역할:
#   transformers  : Hugging Face의 사전학습 모델(BERT, GPT, VL 모델 등) 로드/사용
#   accelerate    : 멀티GPU, mixed precision 등 학습 가속 유틸리티
#   peft          : LoRA 등 파라미터 효율적 파인튜닝(PEFT) 기법 제공
#   bitsandbytes  : 4bit/8bit 양자화(quantization)를 통한 메모리 절약
#   datasets      : Hugging Face 데이터셋 로드 유틸리티
#   pillow        : 이미지 파일 열기/변환 (PIL 라이브러리)
#   pandas        : CSV 등 테이블 형태 데이터 처리
!pip -q install "transformers>=4.43.2,<5.0.0" "accelerate>=0.34.2" "peft>=0.13.2" "bitsandbytes>=0.43.3" datasets pillow pandas --upgrade 



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# 데이터 준비

개발에 필요한 데이터를 준비합니다.

- `train.csv`, `train` 폴더
- `test.csv`, `test` 폴더
- `sample_submission.csv`

데이터를 압축 해제하는데 몇 분 정도의 시간이 소요됩니다.

#### 실습 참고 내용

    챕터 2-2 합성 데이터 실습

# 라이브러리, 데이터, 설정

In [2]:
# ================================================================
# [2단계] 라이브러리 임포트 및 기본 설정
# ================================================================

# --- 표준 라이브러리 ---
import os       # 파일/디렉토리 경로 관리
import re       # 정규표현식 (텍스트 패턴 매칭)
import math     # 수학 함수 (ceil 등 올림 계산에 사용)
import random   # 랜덤 시드 고정용
import gc

# --- 데이터 처리 ---
import pandas as pd          # CSV 파일 읽기 및 DataFrame 조작
from PIL import Image        # 이미지 파일 열기/변환 (Pillow 라이브러리)

# --- PyTorch 핵심 ---
from torch.utils.data import Dataset, DataLoader  # 커스텀 데이터셋 & 배치 로딩
from dataclasses import dataclass                  # 데이터 클래스 데코레이터 (Collator용)
import torch                                       # PyTorch 프레임워크
from typing import Dict, List, Any                 # 타입 힌트 (코드 가독성 향상)

# --- Hugging Face Transformers ---
from transformers import (
    AutoModelForVision2Seq,            # 이미지→텍스트 생성 모델 (VQA, 캡셔닝 등)
    AutoProcessor,                     # 모델에 맞는 전처리기 (토크나이저 + 이미지 처리)
    BitsAndBytesConfig,                # 양자화(4bit/8bit) 설정 클래스
    get_linear_schedule_with_warmup,    # 학습률 스케줄러 (워밍업 후 선형 감소)
)

# --- PEFT (Parameter-Efficient Fine-Tuning) ---
from peft import (
    LoraConfig,                    # LoRA 하이퍼파라미터 설정
    get_peft_model,                # 기존 모델에 LoRA 어댑터를 장착
    prepare_model_for_kbit_training,  # 양자화된 모델을 학습 가능 상태로 준비
)
from tqdm import tqdm  # 진행 상황 표시 막대 (프로그레스바)

# ── 이미지 픽셀 제한 해제 ──
# PIL은 기본적으로 초대형 이미지 로드 시 보안상 제한을 걸어둡니다.
# 대회 데이터에 고해상도 이미지가 포함될 수 있으므로 제한을 해제합니다.
Image.MAX_IMAGE_PIXELS = None

# ── 디바이스 설정 ──
# GPU(CUDA)가 사용 가능하면 "cuda", 아니면 "cpu"를 사용합니다.
# 딥러닝 학습 시 GPU를 사용해야 속도가 수십~수백 배 빨라집니다.
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# ── 하이퍼파라미터 및 모델 설정 ──
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"  # 사용할 사전학습 모델 (Qwen2.5 Vision-Language 3B)
IMAGE_SIZE = 384    # 입력 이미지 해상도 (384×384 픽셀로 리사이즈)
MAX_NEW_TOKENS = 8  # 모델이 생성할 최대 토큰 수 (a/b/c/d 한 글자만 필요)
SEED = 42           # 재현성을 위한 랜덤 시드 (모든 실행에서 같은 결과를 보장)

# 랜덤 시드 고정: Python, PyTorch CPU, PyTorch GPU 모두 동일 시드 적용
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── 데이터셋 로드 ──
# train.csv : 학습 데이터 (이미지 경로, 질문, 선택지 a~d, 정답 포함)
# test.csv  : 테스트 데이터 (정답 없음 → 모델이 예측해야 함)
train_df = pd.read_csv("train.csv")
test_df  = pd.read_csv("test.csv")

# ── 학습 데이터 샘플링 ──
# 전체 학습 데이터 중 200개만 랜덤 추출하여 빠르게 실험합니다.
# 실제 성능을 높이려면 이 숫자를 늘리거나 전체 데이터를 사용하세요.
# random_state=SEED : 매번 같은 200개가 선택되도록 보장
train_df = train_df.sample(n=1000, random_state=SEED).reset_index(drop=True)


Device: cuda


# 모델, Processor

7.5GB 정도의 모델 다운로드가 진행됩니다. 10~20분 정도가 소요됩니다.

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - LoRA 구현 : LoraConfig()

In [3]:
# ================================================================
# [3단계] 모델 로드: 양자화 + 프로세서 + LoRA 적용
# ================================================================

# ── (1) 4bit 양자화 설정 ──
# 양자화(Quantization)란?
#   모델의 가중치(weight)를 32bit → 4bit로 압축하여 GPU 메모리를 크게 절약하는 기법.
#   예) 3B 모델: 32bit 기준 ~12GB → 4bit 기준 ~3GB로 축소
#   정확도 손실을 최소화하면서 메모리를 약 1/4로 줄입니다.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,               # 4bit 양자화 활성화
    bnb_4bit_use_double_quant=True,   # 이중 양자화: 양자화 상수도 다시 양자화하여 추가 메모리 절약
    bnb_4bit_quant_type="nf4",        # NF4(NormalFloat4): 정규분포에 최적화된 4bit 데이터 타입
    bnb_4bit_compute_dtype=torch.float16,  # 연산 시에는 float16 정밀도로 계산 (속도↑)
)

# ── (2) 프로세서(Processor) 로드 ──
# 프로세서 = 토크나이저(텍스트→토큰) + 이미지 전처리기(이미지→텐서)를 하나로 묶은 것.
# VLM(Vision-Language Model)은 텍스트와 이미지를 동시에 입력받으므로,
# 두 가지 전처리를 함께 수행하는 프로세서가 필요합니다.
MIN_PIXELS = 256 * 28 * 28  # 약 20만 픽셀
MAX_PIXELS = 1024 * 28 * 28 # 약 100만 픽셀

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_PIXELS,  # 최소 픽셀 수 (384×384 = 147,456)
    max_pixels=MAX_PIXELS,  # 최대 픽셀 수 (동일하게 고정 → 모든 이미지 동일 크기)
    trust_remote_code=True,  # 모델 저장소의 커스텀 코드 실행 허용
)

# ── (3) 사전학습 모델 로드 ──
# Qwen2.5-VL-3B-Instruct: 30억 개 파라미터를 가진 Vision-Language 모델.
# 이미지를 보고 질문에 답하는 VQA(Visual Question Answering) 능력을 갖추고 있습니다.
# quantization_config : 위에서 설정한 4bit 양자화를 적용하여 로드
# device_map="auto"  : 모델을 GPU 메모리에 자동으로 배치
base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,   # 4bit 양자화 적용
    device_map="auto",                # GPU/CPU에 자동 분배
    trust_remote_code=True,           # 커스텀 코드 실행 허용
)

# ── (4) 양자화 모델을 학습 가능 상태로 전환 ──
# 양자화된 모델은 기본적으로 추론(inference) 전용입니다.
# prepare_model_for_kbit_training()이 학습에 필요한 설정을 자동으로 해줍니다:
#   - 입력 임베딩의 requires_grad 해제
#   - LayerNorm을 float32로 유지 (학습 안정성)
base_model = prepare_model_for_kbit_training(base_model)

# gradient_checkpointing : 역전파 시 중간 결과를 저장하지 않고 필요할 때 재계산.
# GPU 메모리를 절약하는 대신 학습 시간이 약간 늘어납니다. (메모리↓, 속도↓)
base_model.gradient_checkpointing_enable()

# ── (5) LoRA (Low-Rank Adaptation) 설정 ──
# LoRA란?
#   전체 모델의 가중치를 수정하지 않고, 작은 어댑터(adapter)만 추가로 학습하는 기법.
#   비유: 원본 책(사전학습 모델)은 그대로 두고, 포스트잇(어댑터)만 붙이는 것!
#   전체 파라미터의 약 1~2%만 학습하므로 메모리와 시간이 크게 절약됩니다.
lora_config = LoraConfig(
    r=32,                # LoRA 랭크(rank): 어댑터의 크기. 클수록 표현력↑ 메모리↑
    lora_alpha=64,      # LoRA 스케일링 계수: alpha/r 비율로 어댑터 출력을 조절
    lora_dropout=0.1,  # 드롭아웃 비율: 과적합(overfitting) 방지용
    bias="none",        # 바이어스는 학습하지 않음 (메모리 절약)
    target_modules=[    # LoRA를 적용할 레이어 목록 (Transformer의 핵심 레이어들)
        "q_proj",       # Query 프로젝션   ─┐
        "k_proj",       # Key 프로젝션     ├─ Self-Attention 레이어
        "v_proj",       # Value 프로젝션   │
        "o_proj",       # Output 프로젝션  ─┘
        "gate_proj",    # Gate 프로젝션    ─┐
        "up_proj",      # Up 프로젝션      ├─ Feed-Forward Network (FFN) 레이어
        "down_proj",    # Down 프로젝션    ─┘
    ],
    task_type="CAUSAL_LM",  # 태스크 유형: 인과적 언어 모델 (다음 토큰 예측)
)

# ── (6) PEFT 모델 생성 ──
# 기존 base_model에 LoRA 어댑터를 장착하여 학습 가능한 PEFT 모델을 생성합니다.
model = get_peft_model(base_model, lora_config)

# 학습 가능한 파라미터 수를 출력합니다.
# 예시 출력: "trainable params: 6,815,744 || all params: 3,762,xxx,xxx || trainable%: 0.18%"
# → 전체의 약 0.18%만 학습하면 되므로 매우 효율적!
model.print_trainable_parameters()


The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
c:\Users\SSAFY\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\models\auto\modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
W0403 08:30:07.105000 30252 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

c:\Users\SSAFY\AppData\Local\Programs\Python\Python311\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


trainable params: 74,305,536 || all params: 3,828,928,512 || trainable%: 1.9406


# 프롬프트 템플릿

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - 프롬프트 템플릿 : convert_to_chatml(), formatting_prompts_func()

In [4]:
# ================================================================
# [4단계] 프롬프트 템플릿 정의
# ================================================================
# LLM(대규모 언어 모델)에게 원하는 형식으로 답변을 유도하기 위해
# 체계적인 프롬프트(지시문)를 구성합니다.

# ── 시스템 지시사항 ──
# 모델의 역할과 출력 형식을 명확히 정의합니다.
# "한 글자만 출력하라"고 강하게 제한해야 불필요한 설명 없이 답만 나옵니다.
SYSTEM_INSTRUCT = (
    "You are a professional visual reasoning assistant. "
    "Analyze the provided image meticulously and solve the given multiple-choice question. "
    "Follow these steps: 1. Describe key visual elements. 2. Reason through the question. "
    "3. Conclude with the final answer."
)

# ── 사용자 프롬프트 생성 함수 ──
# 질문(question)과 4개 선택지(a, b, c, d)를 받아
# 모델이 이해하기 좋은 형태의 프롬프트 텍스트로 변환합니다.
# 마지막 줄의 한국어 지시는 답변 형식을 한 번 더 강조하는 역할입니다.
def build_mc_prompt(question, a, b, c, d):
    return (
        f"Question: {question}\n"
        f"Options:\n(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "Please reason step-by-step before arriving at the final answer. "
        "At the very end of your response, provide the final answer in the format: 'Final Answer: [letter]'."
    )


# Custom Dataset, Collator

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - TensorDataset()

    챕터 5-2 데이터 생성 및 파인튜닝 (향후 학습 분량)
    - IntentDataset()

In [6]:
# ================================================================
# [5단계] 커스텀 데이터셋(Dataset)과 데이터 콜레이터(Collator) 정의
# ================================================================
# PyTorch의 Dataset/DataLoader 구조:
#   Dataset  → 개별 데이터 1건을 어떻게 가져올지 정의 (__getitem__)
#   Collator → 여러 건의 데이터를 하나의 배치(batch)로 묶는 방법 정의
#   DataLoader → Dataset에서 데이터를 꺼내 Collator로 배치를 만들어 반복 제공

# ── 커스텀 데이터셋 클래스 ──
# VQA(Visual Question Answering) 다지선다(Multiple Choice) 데이터셋
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True):
        """
        Args:
            df (DataFrame): 이미지 경로, 질문, 선택지, 정답이 담긴 데이터프레임
            processor: Hugging Face 프로세서 (토크나이저 + 이미지 전처리기)
            train (bool): True면 정답(answer) 포함 → 학습용, False면 제외 → 추론용
        """
        self.df = df.reset_index(drop=True)  # 인덱스를 0부터 재정렬
        self.processor = processor
        self.train = train

    def __len__(self):
        """데이터셋 전체 길이 반환 (DataLoader가 반복 횟수를 알기 위해 필요)"""
        return len(self.df)

    def __getitem__(self, i):
        """
        i번째 데이터를 가져와 모델 입력 형태(ChatML 메시지)로 변환합니다.

        ChatML 메시지 구조:
          1) system  : 모델의 역할/규칙 정의
          2) user    : 이미지 + 질문 전달
          3) assistant (학습 시만) : 정답 제공 → 모델이 이 답을 생성하도록 학습
        """
        row = self.df.iloc[i]

        # 이미지를 RGB 형식으로 로드 (흑백/투명 이미지도 3채널 RGB로 통일)
        img = Image.open(row["path"]).convert("RGB")

        # CSV에서 질문과 선택지를 문자열로 추출
        q = str(row["question"])
        a, b, c, d = str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])

        # 프롬프트 텍스트 생성
        user_text = build_mc_prompt(q, a, b, c, d)

        # ChatML 형식의 메시지 리스트 구성
        messages = [
            # 시스템 메시지: 모델의 행동 규칙을 설정
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
            # 사용자 메시지: 이미지와 질문을 함께 전달
            {"role": "user", "content": [
                {"type": "image", "image": img},    # 이미지 입력
                {"type": "text", "text": user_text}  # 질문 + 선택지 텍스트
            ]}
        ]

        # 학습 모드일 때만 정답(assistant 메시지)을 추가합니다.
        # → 모델은 이 정답을 "다음 토큰"으로 예측하도록 학습됩니다.
        if self.train:
            gold = str(row["answer"]).strip().lower()  # 정답을 소문자로 정리
            messages.append({"role": "assistant", "content": [{"type": "text", "text": gold}]})

        return {"messages": messages, "image": img}


# ── 데이터 콜레이터 (Data Collator) ──
# 배치(batch)를 구성하는 역할을 합니다.
# 여러 개의 데이터 샘플을 모아 하나의 텐서(tensor) 묶음으로 변환합니다.
@dataclass
class DataCollator:
    """
    DataLoader가 배치를 만들 때 호출하는 함수 객체.
    개별 샘플의 메시지를 프로세서로 인코딩하여 모델 입력 텐서를 생성합니다.
    """
    processor: Any       # Hugging Face 프로세서
    train: bool = True   # 학습 모드 여부

    def __call__(self, batch):
        """
        Args:
            batch (list): VQAMCDataset.__getitem__()의 반환값 리스트
        Returns:
            dict: input_ids, attention_mask, pixel_values, labels 등의 텐서
        """
        texts, images = [], []
        for sample in batch:
            messages = sample["messages"]
            img = sample["image"]

            # apply_chat_template: 메시지 리스트를 모델이 이해하는 텍스트 형식으로 변환
            # 예: "<|im_start|>system\n...\n<|im_end|>\n<|im_start|>user\n..."
            text = self.processor.apply_chat_template(
                messages,
                tokenize=False,              # 텍스트 문자열만 생성 (토큰화는 아래에서)
                add_generation_prompt=False   # 학습 시에는 생성 프롬프트 불필요
            )
            texts.append(text)
            images.append(img)

        # 텍스트와 이미지를 동시에 인코딩하여 모델 입력 텐서로 변환
        # padding=True : 배치 내 길이가 다른 시퀀스를 최대 길이에 맞춰 패딩
        # return_tensors="pt" : PyTorch 텐서로 반환
        enc = self.processor(
            text=texts,
            images=images,
            padding=True,
            truncation=True,  # 메모리 안전장치 추가
            max_length=2048,   # 적절한 길이 제한
            return_tensors="pt"
        )

        # 학습 시 labels를 설정합니다.
        # labels = input_ids의 복사본으로, 모델은 각 위치에서 다음 토큰을 예측합니다.
        # (Causal LM의 학습 원리: 이전 토큰들로 다음 토큰을 맞추는 것)
        if self.train:
            enc["labels"] = enc["input_ids"].clone()

        return enc


# DataLoader

#### 실습 참고 내용

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 데이터로더 정의 : DataLoader()

In [7]:
# ================================================================
# [6단계] DataLoader 생성 (학습/검증 데이터 분리)
# ================================================================

# ── 학습/검증 데이터 분리 ──
# 전체 학습 데이터의 90%는 학습(train), 10%는 검증(validation)용으로 나눕니다.
# 검증 데이터: 학습에 사용되지 않으며, 모델이 새로운 데이터에 얼마나 잘 동작하는지 평가합니다.
# → 과적합(overfitting) 여부를 판단하는 지표로 활용됩니다.
split = int(len(train_df) * 0.9)  # 200개 × 0.9 = 180개 학습, 20개 검증
train_subset, valid_subset = train_df.iloc[:split], train_df.iloc[split:]

# ── Dataset 객체 생성 ──
# 위에서 정의한 VQAMCDataset 클래스로 데이터를 래핑합니다.
train_ds = VQAMCDataset(train_subset, processor, train=True)   # 학습용 (정답 포함)
valid_ds = VQAMCDataset(valid_subset, processor, train=True)   # 검증용 (정답 포함 → 손실 계산)

# ── DataLoader 생성 ──
# DataLoader는 Dataset에서 데이터를 배치(batch) 단위로 꺼내 모델에 전달합니다.
#   batch_size=1   : 한 번에 1개씩 처리 (GPU 메모리 절약, 대신 Gradient Accumulation 사용)
#   shuffle=True   : 학습 데이터를 매 에폭마다 랜덤하게 섞음 (학습 다양성↑)
#   collate_fn     : 배치를 묶는 방식을 위에서 정의한 DataCollator로 지정
#   num_workers=0  : 데이터 로딩 프로세스 수 (0 = 메인 프로세스에서 직접 로딩)
train_loader = DataLoader(
    train_ds, batch_size=1, shuffle=True,
    collate_fn=DataCollator(processor, True), num_workers=0,
    pin_memory=False    # [추가] 메모리 고정 기능을 꺼서 충돌 가능성을 줄입니다.
)
valid_loader = DataLoader(
    valid_ds, batch_size=1, shuffle=False,
    collate_fn=DataCollator(processor, True), num_workers=0,
    pin_memory=False    # [추가] 메모리 고정 기능을 꺼서 충돌 가능성을 줄입니다.
)


# fine-tuning

- 200개만 학습 : 10~20분 소요

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - 모델 정의 : SimpleMLP(), SequentialMLP()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [8]:
# ================================================================
# [7단계] 파인튜닝 (Fine-Tuning) 학습 루프
# ================================================================
from tqdm.auto import tqdm

# 모델을 GPU로 이동
model = model.to(device)

# ── Gradient Accumulation 설정 ──
# GPU 메모리가 부족하여 큰 배치를 한 번에 처리할 수 없을 때 사용합니다.
# batch_size=1이지만, 4번의 순전파(forward) 결과를 누적한 후 1번 역전파(backward)합니다.
# 효과적으로 batch_size=4와 동일한 학습 효과를 얻습니다.
GRAD_ACCUM = 4
EPOCHS = 3  # [추가] 에폭 수 변수화

# ── 옵티마이저 (Optimizer) ──
# AdamW: Adam + Weight Decay (가중치 감쇠)를 적용한 옵티마이저
# lr=1e-4 : 학습률(Learning Rate). 파라미터를 한 번에 얼마나 업데이트할지 결정.
#   → 너무 크면 발산, 너무 작으면 학습이 느림. 1e-4는 LoRA 학습에 자주 쓰이는 값.
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

# ── 학습률 스케줄러 ──
# 학습률을 점진적으로 조절하는 전략:
#   1) 워밍업(warmup): 처음 3%는 학습률을 0에서 1e-4까지 서서히 올림 (안정적 시작)
#   2) 선형 감소(linear decay): 이후 학습률을 서서히 0으로 줄임
# 비유: 자동차를 처음에 서서히 가속하고, 목적지에 가까워지면 서서히 감속하는 것
num_training_steps = EPOCHS * math.ceil(len(train_loader) / GRAD_ACCUM)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(num_training_steps * 0.03),  # 워밍업 스텝 수 (전체의 3%)
    num_training_steps=num_training_steps               # 전체 학습 스텝 수
)

# ── Mixed Precision 스케일러 ──
# GradScaler: float16 연산 시 발생할 수 있는 언더플로우(너무 작은 수) 문제를 방지.
# 기울기(gradient)를 적절히 스케일링하여 수치 안정성을 보장합니다.
scaler = torch.cuda.amp.GradScaler(enabled=True)

# ════════════════════════════════════════════════
# 학습 루프 시작
# ════════════════════════════════════════════════
global_step = 0

# epoch: 전체 학습 데이터를 1번 순회하는 단위. 여기서는 1 에폭만 수행합니다.
for epoch in range(3):
    model.train() # [추가] 확실하게 학습 모드 명시
    running = 0.0  # 누적 손실(loss) 추적 변수
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1} [train]", unit="batch")

    for step, batch in enumerate(progress_bar, start=1):
        # (a) 배치 데이터를 GPU로 이동
        batch = {k: v.to(device) for k, v in batch.items()}

        # (b) Mixed Precision: bfloat16으로 순전파하여 메모리 절약 & 속도 향상
        #     autocast 블록 안에서는 연산이 자동으로 bfloat16 정밀도로 수행됩니다.
        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            outputs = model(**batch)           # 순전파(forward pass)
            loss = outputs.loss / GRAD_ACCUM   # 손실을 누적 횟수로 나눔

        # (c) 역전파(backward): 기울기(gradient)를 계산합니다.
        #     scaler.scale()은 기울기를 스케일링하여 언더플로우를 방지합니다.
        scaler.scale(loss).backward()
        running += loss.item()  # 손실 값 누적 (로깅용)

        # (d) Gradient Accumulation: GRAD_ACCUM 스텝마다 실제 파라미터 업데이트
        if step % GRAD_ACCUM == 0:
            scaler.step(optimizer)           # 옵티마이저로 파라미터 업데이트
            scaler.update()                  # 스케일러 상태 업데이트
            optimizer.zero_grad(set_to_none=True)  # 기울기 초기화 (메모리 효율적)
            scheduler.step()                 # 학습률 조정
            global_step += 1

            # [추가] 10번 업데이트(40 배치)마다 메모리 강제 청소
            if global_step % 10 == 0:
                torch.cuda.empty_cache()
                gc.collect()

            # 현재 평균 손실을 프로그레스바에 표시
            avg_loss = running / GRAD_ACCUM
            progress_bar.set_postfix({"loss": f"{avg_loss:.3f}"})
            running = 0.0  # 누적 손실 초기화

        # [추가] 매 배치 종료 후 대형 객체 삭제 (프리징 방지 핵심)
        del batch, outputs, loss

    # ════════════════════════════════════════════════
    # 검증(Validation) 루프
    # ════════════════════════════════════════════════
    # model.eval() : 드롭아웃 비활성화, 배치정규화 고정 → 추론 모드로 전환
    model.eval()
    val_loss = 0.0
    val_steps = 0

    # torch.no_grad() : 기울기 계산을 비활성화하여 메모리 절약 & 속도 향상
    # 검증은 파라미터를 업데이트하지 않으므로 기울기가 필요 없습니다.
    with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
        for vb in tqdm(valid_loader, desc=f"Epoch {epoch+1} [valid]", unit="batch"):
            vb = {k: v.to(device) for k, v in vb.items()}
            val_loss += model(**vb).loss.item()  # 검증 손실 누적
            val_steps += 1
            del vb # [추가] 검증 시에도 삭제

    # 에폭 종료 시 검증 손실 출력
    # 학습 손실보다 검증 손실이 높으면 과적합(overfitting) 가능성이 있습니다.
    # [수정] ZeroDivisionError 방지 및 결과 출력
    if val_steps > 0:
        print(f"[Epoch {epoch+1}] valid loss {val_loss/val_steps:.4f}")
    else:
        print(f"[Epoch {epoch+1}] Validation skipped or no loss returned.")

    # model.train() : 다시 학습 모드로 전환 (드롭아웃 활성화)
    model.train()

# ── 학습 완료 후 모델 저장 ──
# LoRA 어댑터 가중치와 프로세서 설정을 함께 저장합니다.
# 나중에 이 경로에서 모델을 다시 불러올 수 있습니다.
SAVE_DIR = "checkpoints/qwen2_5_vl_3b_lora"
model.save_pretrained(SAVE_DIR)       # LoRA 어댑터 가중치 저장
processor.save_pretrained(SAVE_DIR)   # 프로세서(토크나이저) 설정 저장
print("Saved:", SAVE_DIR)


C:\Users\SSAFY\AppData\Local\Temp\ipykernel_30252\2665758396.py:37: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=True)


Epoch 1 [train]:   0%|          | 0/900 [00:00<?, ?batch/s]

C:\Users\SSAFY\AppData\Local\Temp\ipykernel_30252\2665758396.py:56: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16):
c:\Users\SSAFY\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Epoch 1 [valid]:   0%|          | 0/100 [00:00<?, ?batch/s]

[Epoch 1] valid loss 6.5929


...

Epoch 2 [valid]:   0%|          | 0/100 [00:00<?, ?batch/s]

[Epoch 2] valid loss 6.5896


Epoch 3 [train]:   0%|          | 0/900 [00:00<?, ?batch/s]

Epoch 3 [valid]:   0%|          | 0/100 [00:00<?, ?batch/s]

[Epoch 3] valid loss 6.5889
Saved: checkpoints/qwen2_5_vl_3b_lora


# inference

30분~1시간 소요

#### 실습 참고 내용

    챕터4-1 RAG 기반 Customer Service AI 에이전트 개발
    - 데이터 파서 : langchain_core.output_parsers(), StrOutputParser()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [9]:
# ================================================================
# [8단계] 추론 (Inference) - 테스트 데이터에 대한 예측
# ================================================================

# ── 데이터 파서 함수 ──
# 모델이 생성한 텍스트에서 정답 선지(a/b/c/d)를 추출합니다.
# 모델이 항상 깔끔하게 한 글자만 출력하지는 않으므로,
# 여러 가지 경우를 처리하는 후처리(post-processing) 로직이 필요합니다.
def extract_choice(text: str) -> str:
    """모델 출력에서 최종 정답(a/b/c/d)을 추출합니다."""
    # 1순위: Final Answer: [알파벳] 형태 찾기
    match = re.search(r"Final Answer:\s*([a-d])", text, re.IGNORECASE)
    if match:
        return match.group(1).lower()
    
    # 2순위: 텍스트 전체에서 가장 마지막에 등장하는 a, b, c, d 찾기
    fallback = re.findall(r"\b([a-d])\b", text.lower())
    if fallback:
        return fallback[-1]

    return "a"  # 어떤 선지도 찾지 못하면 기본값 "a"


# ── 추론 모드 전환 ──
# model.eval() : 드롭아웃 비활성화, 배치정규화 고정 → 일관된 예측 보장
model.eval()
preds = []  # 예측 결과를 저장할 리스트

# [추가] 현재 경로 파악 (이미지 로드용)
BASE_DIR = os.getcwd()

# ── 추론 루프 ──
# 테스트 데이터의 각 행(row)에 대해 모델이 답을 생성합니다.
for i in tqdm(range(len(test_df)), desc="Inference", unit="sample"):
    try:
        row = test_df.iloc[i]

        # 테스트 이미지 로드
        # [추가] 경로 에러 방지를 위한 처리
        img_path = os.path.join(BASE_DIR, row["path"])
        img = Image.open(img_path).convert("RGB")

        # 프롬프트 생성 (질문 + 선택지)
        user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])

        # ChatML 형식의 메시지 구성 (추론 시에는 assistant 메시지 없음)
        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
            {"role": "user", "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": user_text}
            ]}
        ]

        # 메시지를 텍스트로 변환 (추론 시 add_generation_prompt=True)
        # → 끝에 "<|im_start|>assistant\n"를 추가하여 모델이 답을 생성하도록 유도
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

        # 기울기 계산 없이 추론 수행 (메모리 절약)
        # 전체 과정을 autocast로 감싸 처리 속도와 메모리 확보
        with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
            # 연산량 최적화를 위해 inputs 생성과 추론을 하나의 블록으로 묶음
            inputs = processor(text=[text], images=[img], return_tensors="pt").to(device)
            
            out_ids = model.generate(
                **inputs,
                max_new_tokens=512,  # [수정] CoT 추론을 위해 넉넉한 공간 부여
                do_sample=False,     # 샘플링 비활성화 → 가장 확률 높은 토큰 선택 (Greedy Decoding)
                eos_token_id=processor.tokenizer.eos_token_id  # 종료 토큰 ID
            )

        # 생성된 토큰 ID를 텍스트로 디코딩
        # skip_special_tokens=True : 특수 토큰(<|im_start|> 등)을 제외하고 텍스트만 추출
        # [중요] 모델이 새로 생성한 답변(Assistant 답변) 부분만 추출
        generated_ids = out_ids[0][inputs.input_ids.shape[1]:]
        output_text = processor.decode(generated_ids, skip_special_tokens=True)

        # 디버깅용 출력 (필요 시 주석 해제하여 모델 응답 확인)
        # print("output_text:", output_text)
        # print("extract_choice:", extract_choice(output_text))

        # 모델 응답에서 선지(a/b/c/d)를 추출하여 결과 리스트에 추가
        preds.append(extract_choice(output_text))

    except Exception as e:
        print(f"Error at index {i}: {e}")
        preds.append("a") # 에러 발생 시 기본값으로 넘어가서 멈춤 방지

    # 메모리 관리 (RTX 5060 Ti 최적화)
    if 'inputs' in locals(): del inputs
    if 'out_ids' in locals(): del out_ids
    if 'img' in locals(): del img

    if i % 100 == 0:
        torch.cuda.empty_cache()
        gc.collect()

# ── 제출 파일 생성 ──
# 대회 제출 형식에 맞게 id와 예측 답안(answer)을 CSV로 저장합니다.
submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
output_filename = "submission_03.csv"
submission.to_csv(output_filename, index=False)
print(f"Saved: {os.path.join(BASE_DIR, output_filename)}")


Inference:   0%|          | 0/5074 [00:00<?, ?sample/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
c:\Users\SSAFY\AppData\Local\Programs\Python\Python311\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Saved: c:\Users\SSAFY\Desktop\260401_15_2_ai_데이터배포용\submission_03.csv


In [ ]:
# ================================================================
# [참고] 마지막 추론 결과 확인
# ================================================================
# 추론 루프의 마지막 샘플에 대한 모델의 전체 응답 텍스트를 확인합니다.
# 모델이 실제로 어떤 형태로 답을 생성하는지 디버깅/확인 용도입니다.
# 예상 출력: 시스템/사용자 메시지 + 모델이 생성한 선지 (예: "a")
print(output_text)


system
You are a helpful visual question answering assistant. Answer using exactly one letter among a, b, c, or d. No explanation.
user
사진에 보이는 재활용 가능한 아이템은 무엇인가요?
(a) 플라스틱 숟가락
(b) 유리병
(c) 종이 포장지
(d) 금속 캔

정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요.
assistant
a
